# Job Scraper ETL — Jupyter Notebook

**Goal:** Perform ETL (Extract, Transform, Load) on job listings data scraped into SQLite, clean the data, and export to CSV.
**Tech stack:** Python, pandas, sqlite3
**Pipeline Steps:**
1. Extract data from SQLite database.
2. Transform (clean, standardize text, remove duplicates, handle missing values).
3. Load cleaned data into CSV for further analysis or dashboarding.

In [10]:
import sqlite3
import pandas as pd
import re
import logging


# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


# Database and file paths
DB_PATH = 'jobs.db'
OUTPUT_CSV = 'cleaned_jobs.csv'

## 1) Extract — Read Data from SQLite

In [11]:
def extract_data(db_path: str = DB_PATH) -> pd.DataFrame:
	logging.info(f"Extracting data from {db_path} ...")
	conn = sqlite3.connect(db_path)
	df = pd.read_sql_query('SELECT * FROM jobs', conn)
	conn.close()
	logging.info(f"Loaded {len(df)} records from database.")
	return df

## 2) Transform — Clean & Standardize Data

In [12]:
def clean_text(text: str) -> str:
	if pd.isna(text):
		return text
	text = re.sub(r'\s+', ' ', text) # remove multiple spaces
	text = text.strip()
	return text

In [13]:
def transform_data(df: pd.DataFrame) -> pd.DataFrame:
	logging.info("Starting data cleaning and transformation ...")

	# Drop duplicates (based on title, company, location)
	before = len(df)
	df = df.drop_duplicates(subset=['title', 'company', 'location'], keep='first')
	logging.info(f"Removed {before - len(df)} duplicate rows.")

	# Clean text fields
	text_cols = ['title', 'company', 'location', 'summary', 'post_date']
	for col in text_cols:
		if col in df.columns:
			df[col] = df[col].astype(str).apply(clean_text)

	# Handle missing values
	df = df.fillna({'company': 'Unknown', 'location': 'Unspecified'})

	# Standardize date field (example: convert relative terms like '2 days ago')
	df['post_date'] = df['post_date'].replace('Just posted', '0 days ago')

	# Convert relative dates to numeric (e.g., '2 days ago' → 2)
	df['days_ago'] = df['post_date'].str.extract(r'(\d+)').astype(float)

	# Sort by most recent
	if 'days_ago' in df.columns:
		df = df.sort_values('days_ago', ascending=True)

	logging.info(f"Cleaned dataset shape: {df.shape}")
	return df.reset_index(drop=True)

## 3) Load — Export Cleaned Data to CSV

In [14]:
def load_to_csv(df: pd.DataFrame, path: str = OUTPUT_CSV):
	df.to_csv(path, index=False, encoding='utf-8-sig')
	logging.info(f"Cleaned data exported successfully to {path}")

## 4) ETL Pipeline Runner

In [15]:
def run_etl(db_path: str = DB_PATH, output_csv: str = OUTPUT_CSV):
	df_raw = extract_data(db_path)
	df_clean = transform_data(df_raw)
	load_to_csv(df_clean, output_csv)
	return df_clean

## 5) Example Run
Uncomment the next cell to execute the full ETL pipeline.

In [17]:
# Ensure the 'jobs' table exists with sample data
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Create table if it doesn't exist
cursor.execute('''
CREATE TABLE IF NOT EXISTS jobs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    company TEXT,
    location TEXT,
    summary TEXT,
    post_date TEXT
)
''')

# Insert sample data if table is empty
cursor.execute('SELECT COUNT(*) FROM jobs')
if cursor.fetchone()[0] == 0:
    cursor.executemany('''
    INSERT INTO jobs (title, company, location, summary, post_date)
    VALUES (?, ?, ?, ?, ?)
    ''', [
        ('Data Scientist', 'TechCorp', 'New York', 'Analyze data', '2 days ago'),
        ('Software Engineer', 'InnovateX', 'San Francisco', 'Develop software', 'Just posted'),
        ('Analyst', 'BizGroup', 'Chicago', 'Business analysis', '5 days ago')
    ])
    conn.commit()

conn.close()

# Now run the ETL pipeline
df_cleaned = run_etl(DB_PATH, OUTPUT_CSV)
df_cleaned.head()

2025-10-14 12:18:02,423 - INFO - Extracting data from jobs.db ...
2025-10-14 12:18:02,433 - INFO - Loaded 3 records from database.
2025-10-14 12:18:02,433 - INFO - Starting data cleaning and transformation ...
2025-10-14 12:18:02,441 - INFO - Removed 0 duplicate rows.
2025-10-14 12:18:02,467 - INFO - Cleaned dataset shape: (3, 7)
2025-10-14 12:18:02,483 - INFO - Cleaned data exported successfully to cleaned_jobs.csv


,id,title,company,location,summary,post_date,days_ago
0,2,Software Engineer,InnovateX,San Francisco,Develop software,0 days ago,0.0
1,1,Data Scientist,TechCorp,New York,Analyze data,2 days ago,2.0
2,3,Analyst,BizGroup,Chicago,Business analysis,5 days ago,5.0


## 6) Next Steps & Notes
- Add normalization for job titles and locations using lookup tables.
- Integrate with Power BI or Tableau for visualization.
- Automate ETL scheduling via Airflow, cron, or a cloud function.
- Add validation rules (e.g., exclude expired jobs).